# Load a MTRX image for Analysis
Use this notebook to load Scienta Omicron Matrix format SPM data. Your own code can be added to process the data. 

**Author**: Steven R. Schofield  
**Created**: April, 2026

In [ ]:
# For file path operations
import os

In [ ]:
# Path to data processing module
module_path_list = [
    '/Users/steven/academic-iCloud/Python/modules'        # CHANGE THIS TO YOUR OWN PATH WHERE YOU KEEP THESE PYTHON MODULES
] 

module_path = next((p for p in module_path_list if os.path.exists(p)), None)
if not module_path:
    exit("No valid module paths.")
else:
    print('module_path = {}'.format(module_path))
    print(os.listdir(module_path))

In [ ]:
# # Ensure modules are reloaded 
%load_ext autoreload
%autoreload 2

# Import standard modules
import os, sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colormaps  # Import the new colormap module

from natsort import natsorted

# Add custom module path to list
sys.path.append(module_path)

# Import custom module
import SRSML24.data_prep as dp
import SRSML24.data_analysis as da

start_time = dp.current_datetime()

In [ ]:
# The working direcroryt should contain a subfolder 'mtrx' that contains the matrix image data. They can be in subfolders
working_directory = '/Users/steven/Python-data-Hardeep/03-Oct-2025'      # CHANGE THIS TO YOUR OWN PATH WHERE YOU KEEP YOUR MTRX DATA
output_directory = os.path.join(working_directory, 'output')

In [ ]:
job_data_path = dp.create_new_data_path(working_directory, output_directory, include_date=False)
mtrx_path = dp.create_new_data_path(working_directory, 'mtrx', include_date=False)

if not working_directory:
    exit("No valid data paths.")
else:
    print('Top   : {}'.format(working_directory))
    print('Read  : {}'.format(mtrx_path))
    print('Write : {}'.format(job_data_path))

In [ ]:
# Training data
mtrx_file_list, _ = dp.list_files_by_extension(mtrx_path,'Z_mtrx',verbose=False)

In [ ]:
mtrx_file_list = natsorted(mtrx_file_list)  
for i, file in enumerate(mtrx_file_list):
    print(i, file)

In [ ]:
mtrx_file = mtrx_file_list[292]
print('Working with image: {}'.format(mtrx_file))


In [ ]:
imgFU, imgBU, imgFD, imgBD, metadata = dp.load_image_for_processing(mtrx_file)

dp.display_image(imgFU,cmap='gist_heat')

In [ ]:
#imgFU_flat = dp.flatten_image_data(imgFU, flatten_method='poly_xy')
imgFU_flat = dp.flatten_image_data(imgFU, flatten_method='iterate_mask')  # default
#imgFU_flat = dp.flatten_image_data(imgFU, flatten_method='row_mean')
#imgFU_flat = dp.flatten_image_data(imgFU, flatten_method='row_mean_and_slope')

dp.display_image(imgFU_flat,cmap='gist_heat')

In [ ]:
%matplotlib widget
pixel_size = da.pixel_size_from_metadata(imgFU, metadata)
result = da.pick_and_profile(imgFU_flat, cmap='gist_heat',  pixel_size=pixel_size)

In [ ]:
radii, profiles = da.stack_profiles(result['saved'])
fig, ax = da.plot_profiles(radii, profiles)

In [ ]:
da.save_profiles(radii, profiles,filepath=job_data_path,filename=None)

In [ ]:
job_data_path